# Homework 2

In [6]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

## Question 1

In [2]:
from libs.embedder import Embedder

em = Embedder(path="../../models/Xenova/all-MiniLM-L6-v2")

In [4]:
v = em.encode("How does approximate nearest neighbor search work?")

In [5]:
len(v), v[0]

(384, np.float64(-0.020582036807885073))

## Question 2

In [18]:
doc = (next((filter(lambda d: d["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md", documents))))

In [19]:
doc_v = em.encode(doc['content'])

In [20]:
v.dot(doc_v)

np.float64(0.361070280302606)

## Question 3

In [21]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [25]:
chunk_v = em.encode_batch([c['content'] for c in chunks])

In [26]:
scores = chunk_v.dot(v)

In [31]:
max_idx = scores.argmax()

In [32]:
chunks[max_idx]['filename']

'02-vector-search/lessons/07-sqlitesearch-vector.md'

## Question 4

In [34]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["content"])
vindex.fit(chunk_v, chunks)

In [36]:
q = "What metric do we use to evaluate a search engine?"
q_v = em.encode(q)

In [40]:
res = vindex.search(q_v, num_results=1)

res[0]['filename']

'04-evaluation/lessons/05-search-metrics.md'

## Question 5

In [49]:
from minsearch import Index
?Index

Init signature:
Index(
    text_fields,
    keyword_fields=None,
    numeric_fields=None,
    date_fields=None,
    vectorizer_params=None,
)
Docstring:     
A simple search index using TF-IDF and cosine similarity for text fields,
exact matching for keyword fields, and range filters for numeric and date fields.

Attributes:
    text_fields (list): List of text field names to index.
    keyword_fields (list): List of keyword field names to index.
    numeric_fields (list): List of numeric field names to index.
    date_fields (list): List of date field names to index.
    vectorizers (dict): Dictionary of TfidfVectorizer instances for each text field.
    keyword_df (pd.DataFrame): DataFrame containing keyword field data.
    numeric_df (pd.DataFrame): DataFrame containing numeric field data.
    date_df (pd.DataFrame): DataFrame containing date field data.
    text_matrices (dict): Dictionary of TF-IDF matrices for each text field.
    docs (list): List of documents indexed.
Init docs

In [51]:
idx = Index(text_fields=['content'])
idx.fit(chunks)

In [53]:
q = "How do I store vectors in PostgreSQL?"
q_v = em.encode(q)

In [54]:
res_idx = idx.search(q, num_results=5)
res_v = vindex.search(q_v, num_results=5)

In [61]:
set(r['filename'] for r in res_v) - set(r['filename'] for r in res_idx)

{'02-vector-search/lessons/08-pgvector.md'}

## Question 6

In [62]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [64]:
q = "How do I give the model access to tools?"
q_v = em.encode(q)

In [65]:
res_idx = idx.search(q, num_results=5)
res_v = vindex.search(q_v, num_results=5)

In [69]:
rrf_ranked = rrf([res_idx, res_v])

rrf_ranked[0]['filename']

'01-agentic-rag/lessons/13-function-calling.md'